In [1]:
import pandas as pd
import numpy as np

# Load/Read Dataset (from Lab 9)

In [2]:
df = pd.read_csv("data.csv")
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,5/2/2014 0:00,313000.0,3.0,1.50,1340.0,7912.0,1.5,0.0,0.0,3,1340.0,0.0,1955.0,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,5/2/2014 0:00,2384000.0,5.0,2.50,3650.0,9050.0,2.0,0.0,4.0,5,3370.0,280.0,1921.0,0,709 W Blaine St,Seattle,WA 98119,USA
2,5/2/2014 0:00,342000.0,3.0,2.00,1930.0,11947.0,1.0,0.0,0.0,4,1930.0,0.0,1966.0,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,5/2/2014 0:00,420000.0,3.0,2.25,2000.0,8030.0,1.0,0.0,0.0,4,1000.0,1000.0,1963.0,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,5/2/2014 0:00,550000.0,4.0,2.50,1940.0,10500.0,1.0,0.0,0.0,4,1140.0,800.0,1976.0,1992,9105 170th Ave NE,Redmond,WA 98052,USA


# Data Pre-Processing

## 1. Dealing with Null Values

In [4]:
# Check for null values
print("Null values per column:")
print(df.isnull().sum())

Null values per column:
date             0
price            3
bedrooms         5
bathrooms        2
sqft_living      2
sqft_lot         2
floors           2
waterfront       5
view             2
condition        0
sqft_above       3
sqft_basement    3
yr_built         6
yr_renovated     0
street           3
city             4
statezip         3
country          6
dtype: int64


In [5]:
# See percentage of nulls
null_percentage = (df.isnull().sum() / len(df)) * 100
print("Percentage of null values:")
print(null_percentage[null_percentage > 0])

Percentage of null values:
price            0.065217
bedrooms         0.108696
bathrooms        0.043478
sqft_living      0.043478
sqft_lot         0.043478
floors           0.043478
waterfront       0.108696
view             0.043478
sqft_above       0.065217
sqft_basement    0.065217
yr_built         0.130435
street           0.065217
city             0.086957
statezip         0.065217
country          0.130435
dtype: float64


In [6]:
# Handle null values - example: fill numeric columns with median, categorical with mode
# For simplicity, we will fill missing numeric values with median
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill missing categorical/object columns with mode
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "Unknown", inplace=True)

# Verify no nulls remain
print("Null values after handling:")
print(df.isnull().sum().sum())

Null values after handling:
51


C:\Users\UMAIS\AppData\Local\Temp\ipykernel_22792\57254859.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].median(), inplace=True)
C:\Users\UMAIS\AppData\Local\Temp\ipykernel_22792\57254859.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using

## 2. Changing Datatypes of Columns to be "Int"

In [7]:
# Check current data types
print("Data types before conversion:")
print(df.dtypes)

Data types before conversion:
date                 str
price            float64
bedrooms         float64
bathrooms        float64
sqft_living      float64
sqft_lot         float64
floors           float64
waterfront       float64
view             float64
condition          int64
sqft_above       float64
sqft_basement    float64
yr_built         float64
yr_renovated       int64
street               str
city                 str
statezip             str
country              str
dtype: object


In [8]:
# Identify columns that can be converted to integer
# We'll convert columns that are numeric (float64 or int64) but may have decimals,
# and also some columns that are currently object but contain integer-like strings.

# First, handle any float columns: round and convert to int
float_cols = df.select_dtypes(include=['float64']).columns
for col in float_cols:
    df[col] = df[col].round().astype('Int64')  # using pandas nullable integer type

# For integer columns already, ensure they are Int64 (nullable)
int_cols = df.select_dtypes(include=['int64']).columns
for col in int_cols:
    df[col] = df[col].astype('Int64')

# Some object columns that contain only digits (like 'yr_renovated') can be converted
# But careful: many have zeros and missing, we already filled nulls.
possible_int_obj = ['yr_renovated', 'waterfront', 'view', 'condition']
for col in possible_int_obj:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('Int64')

# Convert 'date' to datetime (not int, but good practice)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Check new data types
print("\nData types after conversion to Int:")
print(df.dtypes)


Data types after conversion to Int:
date             datetime64[us]
price                     Int64
bedrooms                  Int64
bathrooms                 Int64
sqft_living               Int64
sqft_lot                  Int64
floors                    Int64
waterfront                Int64
view                      Int64
condition                 Int64
sqft_above                Int64
sqft_basement             Int64
yr_built                  Int64
yr_renovated              Int64
street                      str
city                        str
statezip                    str
country                     str
dtype: object


In [9]:
# Verify conversion by displaying a few rows
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02,313000,3,2,1340,7912,2,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02,2384000,5,2,3650,9050,2,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02,342000,3,2,1930,11947,1,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02,420000,3,2,2000,8030,1,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02,550000,4,2,1940,10500,1,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


In [10]:
# Additional check: ensure no float columns remain
float_remaining = df.select_dtypes(include=['float64', 'float32']).columns
print("Float columns remaining:", list(float_remaining))

Float columns remaining: []


# Summary of changes

In [11]:
print("Dataset shape:", df.shape)
print("No more null values:", df.isnull().sum().sum() == 0)
print("All numeric columns are now Int64 (nullable integer) or datetime.")

Dataset shape: (4600, 18)
No more null values: False
All numeric columns are now Int64 (nullable integer) or datetime.
